# b3_fit

> Phase 22 (`specs/multiagent/phase22/`, experiment B3): the fit. the cells, folds and targets of `b3_cells` (imported here); `simCell` and `gridSim` (the numpy counterpart of a cell through `b3_agents.b3Replay`, every grid point once, cached per θ in `results/multiagent/b3/grid/`); `loss`, `fit`, `heldOut`, `chooseKnob`, `bootTau`, `refit` (the pilot refit of P22-6) and `fitHealth` (FR-B1 to FR-B8).

In [ ]:
#| default_exp b3_fit

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import csv
import multiprocessing as mp
from pathlib import Path

import numpy as np

from explRL.b3_agents import B3_KNOBS, b3Replay
from explRL.b3_cells import (CANDIDATES, CELL, COARSE, FINE_STEP, FIT_CELLS, FOLDS, REUSED, SIM_SEEDS, _target, cellValues,
                             claudeRows, claudeTargets, grid, parseKey, thetaKey)

In [ ]:
#| export
def numpyCell(cell, seed, theta=None):
    "The numpy cell of a fit cell (or a C1 cell, kind 'c1'; Phase 14's fault 'liar': N // 4 liars, as scaleGrid) at one seed, world['b3'] = θ when θ sets a knob."
    from explRL.bridge_grid import c1Cell
    from explRL.colony4 import ARMS4
    from explRL.colony_grid import colonyCell
    from explRL.fix_grid import fixCell
    if cell['kind'] == 'p14':
        c = fixCell('b3', 'baseline', cell['N'], seed, cell['N'] // 4 if cell['fault'] == 'liar' else False, runner='numpy')
    elif cell['kind'] == 'c1':
        c = c1Cell('b3', cell['arm'], seed, cell['fault'], runner='numpy')
    else:
        c = colonyCell('b3', cell['arm'], cell['N'], seed, cell['fault'], 'numpy', arms=ARMS4)
    if thetaKey(theta or {}) != 'default':
        c['cfg']['world']['b3'] = parseKey(thetaKey(theta))
    return c


def simRow(cell, seed, theta=None):
    "One numpy board's row: Phase 14's clueMetrics row, C1's c1Columns, else stage4Columns (FR-B2)."
    from explRL.bridge_mech import c1Columns
    from explRL.colony_mech import stage4Columns
    from explRL.llm_runs import LLMConfig
    cfg = LLMConfig.fromDict(numpyCell(cell, seed, theta)['cfg'])
    rp = b3Replay(cfg, seed)
    return rp.rows[0] if cell['kind'] == 'p14' else c1Columns(cfg, rp) if cell['kind'] == 'c1' else stage4Columns(cfg, rp)


def _one(a):
    cell, seed, theta = a
    return float(simRow(cell, seed, theta)[cell['target']])


def simCell(cell, theta, seeds=SIM_SEEDS):
    "The simulated cell mean of the cell's target over seeds (NaN boards left out)."
    return float(np.nanmean([_one((cell, s, theta)) for s in seeds]))


PLAYS = ('b3_agents', 'b3_fit', 'clue_agents', 'colony_agents', 'bridge_agents', 'clueboard', 'colony_board', 'bridge_board',
         'clue_colony', 'clue_runs', 'colony_trace', 'colony_mech', 'bridge_mech', 'bridge_grid', 'colony_grid', 'llm_agent', 'multi')


def codeHash():
    "The first 12 hex of the SHA-256 of the modules that play a board or compute its row: the grid cache's folder (NFR-3)."
    import hashlib
    import explRL
    d = Path(explRL.__file__).parent
    return hashlib.sha256(b''.join((d / f'{m}.py').read_bytes() for m in PLAYS)).hexdigest()[:12]


def gridSim(cells, thetas, workers=8, seeds=SIM_SEEDS, out=None, play=True):
    """FR-B3: every (θ, cell, seed) once, cached per θ in out (default results/multiagent/b3/grid/<codeHash()>/<key>.csv, so a
    code change never reuses stale boards); -> cell means {key: {cell id: mean}}. gridSim.new is the number of boards this call
    played. workers <= 1 plays in this process; play=False raises instead of playing a missing board."""
    from explRL.train import demo_artifact_dir
    out = Path(out or demo_artifact_dir(None, 'multiagent/b3') / 'grid' / codeHash())
    out.mkdir(parents=True, exist_ok=True)
    have, todo = {}, []
    for th in thetas:
        k, f = thetaKey(th), out / f'{thetaKey(th)}.csv'
        have[k] = {(r['cell'], int(r['seed'])): float(r['value']) for r in csv.DictReader(open(f))} if f.exists() else {}
        todo += [(c, s, parseKey(k)) for c in cells for s in seeds if (c['id'], s) not in have[k]]
    if todo and not play:
        raise RuntimeError(f'{len(todo)} boards are missing from the cache {out}: run tools/b3_run.py grid first')
    if workers > 1 and todo:
        with mp.get_context('spawn').Pool(workers) as pool:
            vals = pool.map(_one, todo, chunksize=4)
    else:
        vals = [_one(a) for a in todo]
    for (c, s, th), v in zip(todo, vals):
        have[thetaKey(th)][(c['id'], s)] = v
    for k in {thetaKey(th) for _, _, th in todo}:
        with open(out / f'{k}.csv', 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['cell', 'seed', 'value'])
            w.writerows([c, s, repr(v)] for (c, s), v in sorted(have[k].items()))
    gridSim.new = len(todo)
    return {k: {c['id']: float(np.nanmean([have[k][(c['id'], s)] for s in seeds])) for c in cells} for k in have}

## Targets, the fit and the folds

In [ ]:
#| export
def loss(theta, cells, sims, targets):
    "Σ over cells of ((sim mean - Claude mean) / se)² at θ (P22-4)."
    m = sims[thetaKey(theta)]
    v = float(sum(((m[c] - targets[c][0]) / targets[c][1]) ** 2 for c in cells))
    if not np.isfinite(v):
        raise ValueError(f'non-finite loss at {thetaKey(theta)}: a simulated or Claude cell mean is not finite')
    return v


def onKnobs(sims, knobs):
    "The grid points of sims that set no knob outside `knobs`."
    return {k: v for k, v in sims.items() if set(parseKey(k)) <= set(knobs)}


def fit(cells, sims, targets, drop=()):
    "FR-B4: the grid point of sims with the least loss over cells minus drop (ties: the first key in sorted order)."
    keep = [c for c in cells if c not in set(drop)]
    return parseKey(min(sorted(sims), key=lambda k: loss(k, keep, sims, targets)))


def heldOut(fold, sims, targets, cells=None):
    "The fold's fit, its loss on the cells the fold left out, and θ_default's loss on them."
    cells, drop = list(cells or targets), FOLDS[fold]
    th, out = fit(cells, sims, targets, drop), [c for c in cells if c in set(drop)]
    lo, d = loss(th, out, sims, targets), loss({}, out, sims, targets)
    return dict(fold=fold, theta=th, loss=lo, default=d, better=lo < d)


def chooseKnob(sims, targets, cells=None):
    "FR-B5: per candidate, the summed held-out loss over S1, S4 and S4b on its grid points; the lowest wins, ties to spread."
    tot = {c: sum(heldOut(f, onKnobs(sims, ('trustOne', c)), targets, cells)['loss'] for f in ('S1', 'S4', 'S4b'))
           for c in CANDIDATES}
    return min(CANDIDATES, key=lambda c: (tot[c], c != 'spread')), tot

In [ ]:
#| export
def bootTau(cells, sims, rows, n=200, seed=0, knobs=None):
    "FR-B6: per knob, the SD of the fitted θ over resamples of Claude boards within cells, at least 0.1."
    vals, rng = {k: v for k, v in cellValues(rows).items() if k in set(cells) and v}, np.random.default_rng(seed)
    knobs = knobs or sorted({k for key in sims for k in parseKey(key)}, key=list(B3_KNOBS).index)
    fits = [fit(list(vals), sims, {c: _target(rng.choice(v, len(v))) for c, v in vals.items()}) for _ in range(n)]
    return {k: max(float(np.std([f.get(k, 0.0) for f in fits])), 0.1) for k in knobs}


def refit(theta0, tau, pilot, sims, params=('trustOne',)):
    """FR-B7, P22-6: the grid point θ (other knobs held at θ0) minimizing Σ_t ((sim_t(θ) - pilot_t) / s_t)² + Σ_p ((θ_p - θ0_p)
    / τ_p)². pilot: [dict(cell=id, value=pilot mean, s=s_t)]; a parameter outside `params` keeps θ0."""
    base = {k: v for k, v in theta0.items() if k not in params and v}
    cand = [k for k in sorted(sims) if {a: b for a, b in parseKey(k).items() if a not in params} == base]
    J = lambda k: (sum(((sims[k][p['cell']] - p['value']) / p['s']) ** 2 for p in pilot) +
                   sum(((parseKey(k).get(q, 0.0) - theta0.get(q, 0.0)) / tau[q]) ** 2 for q in params))
    if not cand or not all(np.isfinite(J(k)) for k in cand):
        raise ValueError('refit: no grid point, or a non-finite pilot value, s or simulated mean')
    return parseKey(min(cand, key=J))


def pilotScale(rows, measure, nBoards, cells=FIT_CELLS):
    "s_t: the Claude per-board SD of a measure, pooled over the fit cells of that measure, over √(pilot boards)."
    v = [x for c in cells if c['target'] == measure for x in [np.array(cellValues(rows, [c])[c['id']])] if len(x) > 1]
    return float(np.sqrt(sum(((x - x.mean()) ** 2).sum() for x in v) / sum(len(x) - 1 for x in v)) / np.sqrt(nBoards))


def fitHealth(fits, held, knobs, chosen):
    """FR-B8: bound hits (a fitted knob at the top of its range, in any fit: the range may be too narrow; P22-21) and H22-3 per
    candidate (held-out loss below θ_default's in at least 2 of S1, S4 and S4b); ok when no bound hit and H22-3 holds for the
    chosen knob (P22-11, P22-18). A knob fitted to 0 (the behaviour off) is reported in atZero and does not stop the run.
    fits {name: θ}; held {candidate: {fold: heldOut(...)}}."""
    hits = [(n, k, th.get(k, 0.0)) for n, th in fits.items() for k in knobs if th.get(k, 0.0) >= B3_KNOBS[k][1]]
    zero = [(n, k) for n, th in fits.items() for k in knobs if not th.get(k)]
    h3 = {c: sum(held[c][f]['better'] for f in ('S1', 'S4', 'S4b')) >= 2 for c in held}
    return dict(boundHits=hits, atZero=zero, H22_3=h3, ok=not hits and bool(h3.get(chosen)))

## Tests

In [ ]:
# test_b3_fit (AC-5, FR-B1 to FR-B6, FR-B8): the Claude targets, the folds, a planted θ recovered on a tiny grid, the cache,
# chooseKnob's rule and tie, bootTau's floor and fitHealth's flags
import tempfile
rows = claudeRows()
T = claudeTargets(rows)
clean = [c['id'] for c in FIT_CELLS if c['target'] == 'pooledShare']
assert len(T) == 32 and len(clean) == 15 and all(T[c][2] == (5 if CELL[c]['kind'] == 'p14' else 8) for c in T)
means = dict(zip(clean, (0.406, 0.384, 0.341, 0.455, 0.422, 0.378, 0.371, 0.363, 0.361, 0.366, 0.359, 0.357, 0.314, 0.359, 0.230)))
means.update(zip([c for c in T if c not in clean], (0.522, 0.669, 0.250, 0.368, 0.449, 0.554, 0.205, 0.284, 0.375, 0.491,
                                                    0.218, 0.256, 0.289, 0.173, 0.224, 0.428, 0.284)))
assert all(abs(T[c][0] - m) < 0.0005 for c, m in means.items()), {c: T[c][0] for c in T}
left = lambda drop: (sum(c in clean for c in T if c not in drop), sum(c not in clean for c in T if c not in drop))
assert {f: left(d) for f, d in FOLDS.items()} == dict(S1=(7, 13), S4=(13, 7), S4b=(15, 13), P14=(12, 17))
assert left(REUSED['C1']) == (15, 15) and all(c in T for d in list(FOLDS.values()) + [REUSED['C1']] for c in d)
# a planted θ: targets from seeds 3004-3007 at θ*, the fit on seeds 3000-3003 over a 3 x 3 grid recovers θ* within one step
tiny = [CELL[c] for c in ('colony/stage1:colony|independent:8', 'colony/stage1:plain|clean:8', 'colony/stage1:colonyGuards|independent:8')]
g, out = [dict(trustOne=a, spread=b) for a in (0.0, 0.5, 1.0) for b in (0.0, 0.5, 1.0)], tempfile.mkdtemp()
sims = gridSim(tiny, g, workers=0, seeds=range(3000, 3004), out=out)
assert gridSim.new == 9 * 3 * 4 and len(sims) == 9 and 'default' in sims
assert gridSim(tiny, g, workers=0, seeds=range(3000, 3004), out=out) == sims and gridSim.new == 0      # cached: no new boards
star = dict(trustOne=1.0, spread=0.5)
tg = {c['id']: (simCell(c, star, range(3004, 3008)), 0.02, 4) for c in tiny}
got = fit([c['id'] for c in tiny], sims, tg)
assert all(abs(got.get(k, 0) - v) <= 0.5 for k, v in star.items()), got
assert fit(list(tg), sims, {c: (sims['trustOne=1.00,spread=0.50'][c], 0.01, 4) for c in tg}) == star
assert simCell(tiny[0], {}, range(3000, 3004)) == sims['default'][tiny[0]['id']]
# chooseKnob: the lowest summed held-out loss wins; a tie goes to spread
ids = list(T)
mk = lambda off: {c: T[c][0] + off for c in ids}
fake = {'default': mk(0.05), 'trustOne=0.20': mk(0.03), 'spread=0.20': mk(0.02), 'shareAll=0.20': mk(0.0), 'guess=0.10': mk(0.04)}
k, tot = chooseKnob(fake, T, ids)
assert k == 'shareAll' and tot['shareAll'] == 0.0 < tot['spread'] < tot['guess']
fake['spread=0.20'] = mk(0.0)
assert chooseKnob(fake, T, ids)[0] == 'spread'
# bootTau: identical boards give the floor 0.1; fitHealth flags a bound hit and an H22-3 failure
same = [dict(r, contamination=0.3, pooledShare=0.3) for r in rows]
assert bootTau(ids, fake, same, n=20) == dict(trustOne=0.1, spread=0.1, shareAll=0.1, guess=0.1)
h = lambda b: {f: dict(better=x) for f, x in zip(('S1', 'S4', 'S4b'), b)}
H = fitHealth(dict(all=dict(trustOne=0.4, spread=0.3)), dict(spread=h((True, True, False)), guess=h((True, False, False))), ('trustOne', 'spread'), 'spread')
assert H['ok'] and H['H22_3'] == dict(spread=True, guess=False) and not H['boundHits']
H = fitHealth(dict(all=dict(trustOne=1.0, spread=0.3), S1=dict(spread=0.3)), dict(spread=h((True, False, False))), ('trustOne', 'spread'), 'spread')
assert not H['ok'] and H['boundHits'] == [('all', 'trustOne', 1.0)] and H['atZero'] == [('S1', 'trustOne')] and H['H22_3'] == dict(spread=False)
assert fitHealth(dict(all=dict(spread=0.3)), dict(spread=h((True, True, False))), ('trustOne', 'spread'), 'spread')['ok']   # 0 is not a bound hit (P22-21)
try:                                                                        # cache-only: a missing board raises, none is played
    gridSim(tiny, [dict(trustOne=0.2)], workers=0, seeds=range(3000, 3002), out=out, play=False); assert False
except RuntimeError:
    pass
try:                                                                        # a non-finite cell mean never wins the argmin silently
    loss({}, ids, {'default': {c: float('nan') for c in ids}}, T); assert False
except ValueError:
    pass
assert len(codeHash()) == 12 and 'b3_agents' in PLAYS
assert len(grid(('trustOne', 'guess'))) == 36 and len(grid(('trustOne', 'spread'), 'fine')) == 121 and grid(('guess',), 'fine')[-1] == dict(guess=0.5)

In [ ]:
# test_b3_refit (AC-6, FR-B7, P22-6): a pilot equal to θ0's simulation gives θ0; a knob with no pilot measure keeps θ0;
# with a large τ the refit is the pilot-only fit
c0 = 'pilot:colonyGuards|independent:64'
S = {thetaKey(dict(trustOne=a, spread=b)): {c0: 0.1 + 0.4 * a - 0.1 * b} for a in np.round(np.arange(0, 1.01, 0.1), 2) for b in (0.0, 0.3)}
S = {('default' if k == 'trustOne=0.00' else k.replace('trustOne=0.00,', '')): v for k, v in S.items()}
th0, tau = dict(trustOne=0.3, spread=0.3), dict(trustOne=0.1)
assert refit(th0, tau, [dict(cell=c0, value=S[thetaKey(th0)][c0], s=0.05)], S) == th0
assert refit(th0, tau, [], S) == th0
far = [dict(cell=c0, value=S['trustOne=0.90,spread=0.30'][c0], s=0.05)]
r1 = refit(th0, tau, far, S)
assert r1['spread'] == 0.3 and 0.3 < r1['trustOne'] < 0.9                  # pulled toward the pilot, spread kept at θ0
assert refit(th0, dict(trustOne=1e6), far, S) == dict(trustOne=0.9, spread=0.3)
assert pilotScale(rows, 'contamination', 1) > pilotScale(rows, 'contamination', 4) > 0

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()